In [1]:
import networkx as nx
import numpy as np
from scipy.io import mmread
import pandas as pd


from sklearn import metrics

import copy
import scipy.sparse as sp
import os
import anndata as ad
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import scanpy as sc
import pandas as pd

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



In [2]:
gene_TCR = sc.read_h5ad('../data/merge_gex_all_donors_all_peptides_meta_for_Leah_dat.h5ad')

/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [3]:
gene_TCR.obs = gene_TCR.obs.reset_index(drop=False)
gene_TCR.obs

,barcode,v_gene_TRA,v_gene_TRB,d_gene_TRA,d_gene_TRB,j_gene_TRA,j_gene_TRB,c_gene_TRA,c_gene_TRB,cdr3_TRA,...,A2402_AYSSAGASI_NC,B0702_GPAESAAGL_NC,NR(B0801)_AAKGRGAAL_NC,antigen,peptide,orig_ix,n_counts,log_counts,n_genes,mt_fraction
0,ACTTGTTTCCTTAATC-36,TRAV12-2,TRBV7-2,None,None,TRAJ42,TRBJ2-7,TRAC,TRBC2,CAVNIGGGSQGNLIF,...,0.0,0.0,0.0,A0201_GILGFVFTL_Flu-MP_Influenza_binder,GILGFVFTL,0,9221.0,3.964778,2409,0.057369
1,CTCGGGAAGCGATGAC-18,TRAV12-2,TRBV6-3,None,TRBD1,TRAJ33,TRBJ1-2,TRAC,TRBC1,CAARNYQLIW,...,0.0,0.0,0.0,unknown,nan,1,3280.0,3.515874,954,0.055488
2,TCTGGAAGTCACCCAG-15,TRAV19,TRBV30,None,TRBD1,TRAJ17,TRBJ1-6,TRAC,TRBC1,CALKLIKAAGNKLTF,...,0.0,0.0,0.0,A0301_KLGGALQAK_IE-1_CMV_binder,KLGGALQAK,2,3623.0,3.559068,1493,0.077560
3,GTGCTTCTCAGCGATT-4,TRAV19,TRBV11-2,None,TRBD1,TRAJ49,TRBJ1-2,TRAC,TRBC1,CALSEPNTGNQFYF,...,0.0,0.0,0.0,A1101_IVTDFSVIK_EBNA-3B_EBV_binder,IVTDFSVIK,3,1280.0,3.107210,733,0.167969
4,TTTATGCTCTGCCAGG-25,TRAV9-2,TRBV4-3,None,TRBD2,TRAJ23,TRBJ2-3,TRAC,TRBC2,CAWGRNQGGKLIF,...,0.0,0.0,0.0,unknown,nan,4,7320.0,3.864511,1700,0.060792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,TCGGGACAGCCCAATT-5,TRAV21,TRBV6-6,None,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAVDLMKTSYDKVIF,...,0.0,0.0,0.0,unknown,nan,145999,1318.0,3.119915,805,0.110015
145475,CGCGGTAAGCTGATAA-2,TRAV26-2,TRBV4-3,None,TRBD1,TRAJ45,TRBJ1-6,TRAC,TRBC1,CILSRQEGGADGLTF,...,0.0,0.0,0.0,A0301_KLGGALQAK_IE-1_CMV_binder,KLGGALQAK,146000,1440.0,3.158362,906,0.063889
145476,TTCGGTCCACTGTCGG-8,TRAV1-2,TRBV6-4,None,TRBD2,TRAJ33,TRBJ2-1,TRAC,TRBC2,CAVMDSNYQLIW,...,0.0,0.0,0.0,unknown,nan,146001,5430.0,3.734800,1600,0.065009
145477,TTCTTAGCACAACTGT-8,TRAV38-1,TRBV7-8,None,TRBD1,TRAJ43,TRBJ1-4,TRAC,TRBC1,CAFFNNDMRF,...,1.0,0.0,0.0,unknown,nan,146002,5855.0,3.767527,1617,0.057899


In [4]:
ls_index = os.listdir('../5 to 1 Indices')
ls_index

['smart_aligned_v2_train_indices_random_split_2.npy',
 'smart_aligned_v2_train_indices_random_split_4.npy',
 'smart_aligned_v2_train_indices_tcr_split_4.npy',
 'smart_aligned_v2_test_indices_tcr_ab_split_1.npy',
 'smart_aligned_v2_test_indices_random_split_2.npy',
 'smart_aligned_v2_train_indices_tcr_split_3.npy',
 'smart_aligned_v2_train_indices_tcr_ab_split_5.npy',
 'smart_aligned_v2_val_indices_tcr_split_4.npy',
 'smart_aligned_v2_train_indices_tcr_split_1.npy',
 'smart_aligned_v2_test_indices_random_split_3.npy',
 'smart_aligned_v2_test_indices_tcr_ab_split_3.npy',
 'smart_aligned_v2_val_indices_tcr_split_5.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_1.npy',
 'smart_aligned_v2_train_indices_random_split_1.npy',
 'smart_aligned_v2_val_indices_random_split_1.npy',
 'smart_aligned_v2_val_indices_random_split_3.npy',
 'smart_aligned_v2_test_indices_tcr_split_3.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_4.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_5.npy',
 'smart_ali

In [5]:
ref_dat = pd.read_csv('../data/smart_aligned_v2_dataset_reference_5_to_1.csv')
ref_dat

/tmp/ipykernel_1036465/1300321392.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  ref_dat = pd.read_csv('../data/smart_aligned_v2_dataset_reference_5_to_1.csv')


,tcr,peptide,label,tcr_source_dataset,tcr_source_index,peptide_source_dataset,peptide_source_index,donor,binding_tcr
0,CASSLYEQYF,GILGFVFTL,1,10X,0,10X,0,donor1,Y
1,CAWTGTGKIGWDSPLHF,KLGGALQAK,1,10X,2,10X,2,donor1,Y
2,CASSWGGGSHYGYTF,IVTDFSVIK,1,10X,3,10X,3,donor1,Y
3,CASSLYSATGELFF,AVFDRKSDAK,1,10X,5,10X,5,donor1,Y
4,CASSLYSATGELFF,AVFDRKSDAK,1,10X,6,10X,6,donor1,Y
...,...,...,...,...,...,...,...,...,...
431041,CASSFGRGEGEQYF,ELAGIGILTV,0,10X,39940,10X,258,NaN,N
431042,CASSPHFQVDTGELFF,IVTDFSVIK,0,10X,85427,10X,3,NaN,Y
431043,CASSLMRGGTYNSPLHF,FLYALALLL,0,10X,72515,10X,134,NaN,N
431044,CASSVSSTDTQYF,GILGFVFTL,0,10X,133931,10X,0,NaN,Y


In [6]:
batch_correction = pd.read_csv('../data/gex_pca_harmony_all_donor_all_peptide_pair_method.csv')
batch_correction

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,3.036558,0.470259,-1.534361,1.494185,-2.802884,0.413915,-2.406211,1.511487,2.227762,-0.900201,...,-1.173055,0.916518,1.002145,-0.716572,1.147374,1.177967,-0.394466,0.175063,0.101577,-0.650903
1,3.568292,-4.473117,1.128025,-1.154003,-0.340075,-0.331987,-0.711000,0.480116,1.402941,-1.596925,...,0.252037,-0.189764,-0.193532,0.481584,0.704797,-0.477740,-0.321027,-0.303495,-0.785064,-0.610597
2,1.598286,-6.589628,-0.093850,-1.273293,-0.144133,0.581436,1.557961,-0.735445,-2.621554,0.874783,...,-0.040988,-0.404409,-0.550357,0.698925,0.180182,0.012100,-0.074760,0.533318,0.216403,0.333611
3,1.801606,2.185353,-0.062215,0.000600,-0.906292,2.433735,-0.400967,0.401172,-0.236139,0.913400,...,-0.799589,1.570934,0.222576,-0.422355,0.685801,0.413194,-1.090396,0.127215,0.058785,0.415131
4,10.459624,1.670054,-5.228515,5.566212,-5.204177,0.865626,2.290748,-1.135377,1.517275,2.510904,...,-0.023412,1.371591,-0.778464,-0.684442,-0.394926,0.551291,-0.294668,0.061575,-0.203088,-1.092191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,-16.187646,1.508734,-1.991076,-0.155738,0.383084,-0.886498,1.670027,-1.473262,0.665272,-0.767261,...,0.104526,-0.036424,0.123254,-0.486086,-0.272758,-0.008538,-0.298798,-0.302226,-0.595866,0.084635
145475,1.053205,-1.751885,0.171986,-0.500772,0.890879,1.848117,-0.074121,-2.129117,0.646666,-1.529322,...,0.050863,-0.160187,-0.953540,1.035829,-0.226151,-0.638986,-0.062691,0.617543,-0.507099,-1.065667
145476,0.229644,3.210759,-5.414100,-1.201439,0.934827,-0.937627,-0.753186,-0.134533,-0.124110,0.155970,...,-0.065988,0.211532,-0.294399,0.923304,-0.298969,-0.186653,0.899904,-0.126854,0.541319,0.687603
145477,-16.304995,-2.528540,3.613680,0.796308,0.277799,-0.970278,1.526098,-0.509714,0.608519,-0.181751,...,-0.517029,0.294368,-1.016489,0.503258,-0.715753,-0.543770,-0.070787,0.583562,-0.147427,0.412472


In [7]:
ls_index

['smart_aligned_v2_train_indices_random_split_2.npy',
 'smart_aligned_v2_train_indices_random_split_4.npy',
 'smart_aligned_v2_train_indices_tcr_split_4.npy',
 'smart_aligned_v2_test_indices_tcr_ab_split_1.npy',
 'smart_aligned_v2_test_indices_random_split_2.npy',
 'smart_aligned_v2_train_indices_tcr_split_3.npy',
 'smart_aligned_v2_train_indices_tcr_ab_split_5.npy',
 'smart_aligned_v2_val_indices_tcr_split_4.npy',
 'smart_aligned_v2_train_indices_tcr_split_1.npy',
 'smart_aligned_v2_test_indices_random_split_3.npy',
 'smart_aligned_v2_test_indices_tcr_ab_split_3.npy',
 'smart_aligned_v2_val_indices_tcr_split_5.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_1.npy',
 'smart_aligned_v2_train_indices_random_split_1.npy',
 'smart_aligned_v2_val_indices_random_split_1.npy',
 'smart_aligned_v2_val_indices_random_split_3.npy',
 'smart_aligned_v2_test_indices_tcr_split_3.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_4.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_5.npy',
 'smart_ali

In [ ]:

# for i in ls_index:
#     if i.endswith('.npy'):
#         data_ix = np.load(os.path.join('../5 to 1 Indices', i), allow_pickle=True)
#         ref_dat_index = ref_dat.iloc[data_ix]
#         gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
#         TCR_dat = gene_TCR_integration.obs[['barcode','cdr3_TRB']].copy()
#         TCR_dat.columns = ['contig_id','cdr3']
#         v = TCR_dat['contig_id'].values
#         TCR_dat['contig_id'] = [v[ix] +"_" +str(ix) for ix in range(TCR_dat.shape[0])]
#         batch_correction_dat = batch_correction.iloc[ref_dat_index['tcr_source_index'].values].copy().reset_index(drop=True)
#         batch_correction_dat.to_csv(f'./data_10X_5_to_1/gene_batch_correction_{i.split(".")[0]}.csv', index=False)
#         TCR_dat.to_csv(f'./data_10X_5_to_1/TCR_{i.split(".")[0]}.csv', index=False)

/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:183:

In [ ]:
batch_correction_dat

In [ ]:
gene_TCR_integration.obs[['barcode','cdr3_TRB']]

In [ ]:
pd.DataFrame(gene_TCR_integration.X.toarray())

In [ ]:
# python3 Tessa_main.py -tcr ./TCR_dat_for_Tessa_contig_cdr3_barcode.csv -model ./BriseisEncoder/TrainedEncoder.h5 -embeding_vectors ./BriseisEncoder/Atchley_factors.csv -output_TCR res_10X_TCR_emb.csv -output_log res_10X.log -exp ./TCR_dat_for_Tessa_genes_as_rows.csv -output_tessa /results -within_sample_networks FALSE

In [13]:

for i in ls_index:
    if i.endswith('.npy'):
        data_ix = np.load(os.path.join('../5 to 1 Indices', i), allow_pickle=True)
        ref_dat_index = ref_dat.iloc[data_ix]
        gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
        TCR_dat = gene_TCR_integration.obs[['barcode','cdr3_TRB']].copy()
        TCR_dat.columns = ['contig_id','cdr3']
        v = TCR_dat['contig_id'].values
        TCR_dat['contig_id'] = [v[ix] +"_" +str(ix) for ix in range(TCR_dat.shape[0])]
        batch_correction_dat = batch_correction.iloc[ref_dat_index['tcr_source_index'].values].copy().reset_index(drop=True)
        batch_correction_dat = batch_correction_dat.T
        batch_correction_dat.to_csv(f'./data_10X_5_to_1/gene_batch_correction_{i.split(".")[0]}.csv', index=True)
        TCR_dat.to_csv(f'./data_10X_5_to_1/TCR_{i.split(".")[0]}.csv', index=False)

/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/anndata/_core/anndata.py:183:

In [ ]:
TCR_dat

In [ ]:
TCR_dat.shape[0]

In [12]:
with open("run_tessa_commands_5_to_1.sh", "w") as file:
    for i in ls_index:
        if i.endswith('.npy'):
            tessa_command = f'python3 Tessa_main.py -tcr ./data_10X_5_to_1/TCR_{i.split(".")[0]}.csv -model ./BriseisEncoder/TrainedEncoder.h5 -embeding_vectors ./BriseisEncoder/Atchley_factors.csv -output_TCR ./integration_res_5_to_1/res_10X_TCR_emb_{i.split(".")[0]}.csv -output_log res_10X_{i.split(".")[0]}.log -exp ./data_10X_5_to_1/gene_batch_correction_{i.split(".")[0]}.csv -output_tessa /results_5_to_1 -within_sample_networks FALSE\n'
            file.write(tessa_command)   